# 📊 Análisis Exploratorio — Transferencias a Terceros
**Contexto:** Fraude transaccional · Scotiabank Perú · Prevención de Fraude  
**Criterio de extracción:** Código de transacción = 40 · Tipo de cuenta = 1010  
**Monto mínimo:** S/ 1,200.00  
**Fuente:** Monitor (8750) — Archivos Excel mensuales  
**Meses cubiertos:** Enero · Marzo · Abril  

---
> **Indicadores de fraude:**  
> `F` = Fraude confirmado · `G` = Buena (cliente confirmó) · `P` = Pendiente · `D` = Descarte

## 0. Librerías

In [ ]:
import polars as pl
import polars.selectors as cs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from IPython.display import display

# ── Estilo visual ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#3a3f5c',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2a2d3e',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'figure.dpi':       120,
})
PALETTE_CAT  = ['#4fc3f7', '#ef5350', '#66bb6a', '#ffa726', '#ab47bc']
COLOR_FRAUD  = '#ef5350'
COLOR_OK     = '#4fc3f7'
COLOR_ACCENT = '#ffa726'

print(f"Polars  {pl.__version__}")
print(f"Pandas  {pd.__version__}")
print("✅ Librerías cargadas")

## 1. Ingesta — Lectura y unión de los 4 Excel de Monitor

In [ ]:
# ── 1.1  Rutas de los archivos  ────────────────────────────────────────────────
# ⚠️  Ajusta los nombres de archivo según lo que descargaste de Monitor
BASE_DIR = Path(r"C:\Users\s4930359\Data_Herramientas\data\transferencias_terceros")

ARCHIVOS = {
    "Enero" : BASE_DIR / "monitor_transferencias_enero.xlsx",
    "Marzo" : BASE_DIR / "monitor_transferencias_marzo.xlsx",
    "Abril" : BASE_DIR / "monitor_transferencias_abril.xlsx",
    # Si tienes un cuarto mes, agrégalo aquí:
    # "Mes4" : BASE_DIR / "monitor_transferencias_mes4.xlsx",
}

# ── 1.2  Función de lectura con skip=4 + dtype str ────────────────────────────
def leer_monitor_excel(path: Path, mes: str) -> pl.DataFrame:
    """
    Lee un export de Monitor:
      - Salta las 4 primeras filas de header basura
      - Todo llega como string (infer_schema=False)
      - Agrega columna 'mes_origen' para trazabilidad
    """
    df_pd = pd.read_excel(
        path,
        skiprows=4,
        dtype=str,          # TODO como texto
        header=0,
        engine="openpyxl",
    )
    # Limpieza de nombres de columnas
    df_pd.columns = (
        df_pd.columns
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^A-Z0-9_]", "", regex=True)
    )
    df = pl.from_pandas(df_pd)
    df = df.with_columns(pl.lit(mes).alias("MES_ORIGEN"))
    return df

# ── 1.3  Leer y concatenar ────────────────────────────────────────────────────
frames = [leer_monitor_excel(path, mes) for mes, path in ARCHIVOS.items()]
raw = pl.concat(frames, how="diagonal_relaxed")   # diagonal_relaxed maneja columnas distintas entre meses

print(f"Shape total: {raw.shape[0]:,} filas × {raw.shape[1]} columnas")
raw.head(3)

## 2. Inspección de columnas — ¿Qué tenemos?

In [ ]:
# Ver todas las columnas disponibles
for i, col in enumerate(raw.columns, 1):
    print(f"{i:>3}. {col}")

In [ ]:
# Nulos por columna
nulos = (
    raw.select([
        pl.col(c).is_null().sum().alias(c) for c in raw.columns
    ])
    .unpivot(variable_name="columna", value_name="nulos")
    .with_columns((pl.col("nulos") / raw.height * 100).round(2).alias("pct_nulo"))
    .sort("pct_nulo", descending=True)
)
display(nulos.filter(pl.col("nulos") > 0))

## 3. Mapeo de columnas de Monitor

> ⚠️ **Ajusta este diccionario** con los nombres exactos que aparecieron arriba.
> Los nombres entre comillas son los que Monitor suele exportar (pueden variar).

In [ ]:
# ── Mapeo: nombre en Export → nombre estándar interno ────────────────────────
# Ajusta según lo que salió en la celda de columnas de arriba
COL_MAP = {
    # Columna real en Monitor       →  Alias interno
    "FECHA_DE_TRANSACCION"          : "FECHA_TRX",      # AAAAMMDD pegado
    "HORA_DE_LA_TRANSACCION"        : "HORA_TRX",       # HHMMSS  o similar
    "NUMERO_DE_CUENTA"              : "CUENTA",
    "MONTO_DE_TRANSACCION"          : "MONTO",
    "INDICADOR_DE_FRAUDE"           : "INDICADOR",
    "CODIGO_DE_TRANSACCION"         : "COD_TRX",
    "TIPO_DE_CUENTA"                : "TIPO_CUENTA",
    "DESCRIPCION_DEL_COMERCIO"      : "COMERCIO",
    "CODIGO_DE_RESPUESTA"           : "COD_RESPUESTA",
    "NUMERO_DE_TARJETA"             : "TARJETA",
    "NUMERO_DE_ALERTA"              : "ALERTA",
    "NOMBRE_DEL_CLIENTE"            : "CLIENTE",
    # ── agrega o quita según tu caso ──
}

# Renombrar solo las que existen
cols_a_renombrar = {k: v for k, v in COL_MAP.items() if k in raw.columns}
df = raw.rename(cols_a_renombrar)

print(f"Columnas renombradas: {len(cols_a_renombrar)}")
print(list(df.columns))

## 4. Casteo de tipos — Fecha · Hora · Monto

In [ ]:
# ── 4.1  MONTO → Float ───────────────────────────────────────────────────────
df = df.with_columns(
    pl.col("MONTO")
      .str.replace_all(",", "")     # quitar separadores de miles si los hay
      .cast(pl.Float64, strict=False)
      .alias("MONTO")
)

# ── 4.2  FECHA_TRX → Date  (formato AAAAMMDD pegado) ─────────────────────────
df = df.with_columns(
    pl.col("FECHA_TRX")
      .str.strip_chars()
      .str.to_date("%Y%m%d", strict=False)
      .alias("FECHA_TRX")
)

# ── 4.3  HORA_TRX → Time (formato HHMMSS) ────────────────────────────────────
# Monitor suele traer la hora como 'HORA DE LA TRANS.' en formato HHMMSS
df = df.with_columns(
    pl.col("HORA_TRX")
      .str.strip_chars()
      .str.zfill(6)                         # asegura 6 dígitos si viene sin ceros
      .str.to_time("%H%M%S", strict=False)
      .alias("HORA_TRX")
)

# ── 4.4  FECHA_HORA combinada (Datetime) ──────────────────────────────────────
df = df.with_columns(
    (pl.col("FECHA_TRX").cast(pl.Utf8) + " " + pl.col("HORA_TRX").cast(pl.Utf8))
      .str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False)
      .alias("FECHA_HORA")
)

print("Tipos finales:")
print(df.select(["FECHA_TRX", "HORA_TRX", "FECHA_HORA", "MONTO"]).dtypes)
df.select(["FECHA_TRX", "HORA_TRX", "FECHA_HORA", "MONTO"]).head(5)

## 5. Ingeniería de Variables — Flags & Features

In [ ]:
# ── 5.1  Variable objetivo binaria ───────────────────────────────────────────
df = df.with_columns([
    # Fraude binario (solo F=1)
    (pl.col("INDICADOR").str.strip_chars().str.to_uppercase() == "F")
      .cast(pl.Int8).alias("FLAG_FRAUDE"),

    # Clasificado = F, G, D (excluye P y N para métricas limpias)
    pl.col("INDICADOR").str.strip_chars().str.to_uppercase()
      .is_in(["F", "G", "D"]).cast(pl.Int8).alias("FLAG_CLASIFICADO"),

    # Pendiente
    (pl.col("INDICADOR").str.strip_chars().str.to_uppercase() == "P")
      .cast(pl.Int8).alias("FLAG_PENDIENTE"),
])

# ── 5.2  Variables temporales ────────────────────────────────────────────────
df = df.with_columns([
    pl.col("FECHA_TRX").dt.month().alias("MES"),
    pl.col("FECHA_TRX").dt.day().alias("DIA"),
    pl.col("FECHA_TRX").dt.weekday().alias("DIA_SEMANA"),   # 1=Lun … 7=Dom
    pl.col("FECHA_HORA").dt.hour().alias("HORA"),

    # Franja horaria
    pl.when(pl.col("FECHA_HORA").dt.hour().is_between(0, 5)).then(pl.lit("Madrugada"))
      .when(pl.col("FECHA_HORA").dt.hour().is_between(6, 11)).then(pl.lit("Mañana"))
      .when(pl.col("FECHA_HORA").dt.hour().is_between(12, 17)).then(pl.lit("Tarde"))
      .otherwise(pl.lit("Noche")).alias("FRANJA_HORARIA"),

    # Fin de semana
    (pl.col("FECHA_TRX").dt.weekday() >= 6).cast(pl.Int8).alias("FLAG_FIN_SEMANA"),

    # Últimos 5 días del mes
    (pl.col("FECHA_TRX").dt.day() >= 26).cast(pl.Int8).alias("FLAG_FIN_MES"),
])

# ── 5.3  Flags de monto ───────────────────────────────────────────────────────
UMBRAL_ALTO  = 5_000.0
UMBRAL_CRITICO = 10_000.0

df = df.with_columns([
    (pl.col("MONTO") >= UMBRAL_ALTO).cast(pl.Int8).alias("FLAG_MONTO_ALTO"),
    (pl.col("MONTO") >= UMBRAL_CRITICO).cast(pl.Int8).alias("FLAG_MONTO_CRITICO"),
])

# ── 5.4  Velocidad por cuenta (ventana diaria) ───────────────────────────────
vel_cuenta = (
    df.group_by(["CUENTA", "FECHA_TRX"])
      .agg(pl.len().alias("VEL_TRX_DIA_CUENTA"),
           pl.col("MONTO").sum().alias("VEL_MONTO_DIA_CUENTA"))
)
df = df.join(vel_cuenta, on=["CUENTA", "FECHA_TRX"], how="left")

# ── 5.5  Flag velocidad alta (> 3 trx mismo día misma cuenta) ────────────────
df = df.with_columns(
    (pl.col("VEL_TRX_DIA_CUENTA") > 3).cast(pl.Int8).alias("FLAG_VEL_ALTA")
)

# ── 5.6  Z-score de monto por cuenta ─────────────────────────────────────────
stats_cuenta = (
    df.group_by("CUENTA")
      .agg(pl.col("MONTO").mean().alias("_mu_monto"),
           pl.col("MONTO").std().alias("_sd_monto"))
)
df = (
    df.join(stats_cuenta, on="CUENTA", how="left")
      .with_columns(
          ((pl.col("MONTO") - pl.col("_mu_monto")) / (pl.col("_sd_monto") + 1e-9))
            .round(3).alias("ZSCORE_MONTO_CUENTA")
      )
      .drop(["_mu_monto", "_sd_monto"])
)

# Flag anomalía de monto (|z| > 2)
df = df.with_columns(
    (pl.col("ZSCORE_MONTO_CUENTA").abs() > 2.0).cast(pl.Int8).alias("FLAG_ANOMALIA_MONTO")
)

print(f"Shape con features: {df.shape}")
df.select(["FECHA_TRX", "MONTO", "INDICADOR", "FLAG_FRAUDE", "FRANJA_HORARIA",
           "VEL_TRX_DIA_CUENTA", "ZSCORE_MONTO_CUENTA", "FLAG_ANOMALIA_MONTO"]).head(5)

## 6. Vista General — KPIs Ejecutivos

In [ ]:
# ── Solo transacciones clasificadas (F, G, D) para métricas limpias ───────────
df_cal = df.filter(pl.col("FLAG_CLASIFICADO") == 1)

total_trx   = df.height
total_cal   = df_cal.height
total_fraud = df_cal.filter(pl.col("FLAG_FRAUDE") == 1).height
monto_fraud = df_cal.filter(pl.col("FLAG_FRAUDE") == 1)["MONTO"].sum()
fraud_rate  = total_fraud / total_cal * 100 if total_cal > 0 else 0
severidad   = monto_fraud / total_fraud if total_fraud > 0 else 0

print("━" * 55)
print(f"  Total transacciones (dataset)   : {total_trx:>10,}")
print(f"  Transacciones clasificadas       : {total_cal:>10,}")
print(f"  Fraudes confirmados (F)          : {total_fraud:>10,}")
print(f"  Monto total fraudulento (S/)     : {monto_fraud:>13,.2f}")
print(f"  Fraud Rate (sobre clasificadas)  : {fraud_rate:>10.4f} %")
print(f"  Severidad promedio (S/ por caso) : {severidad:>13,.2f}")
print("━" * 55)

In [ ]:
# Distribución por indicador
dist_ind = (
    df.group_by("INDICADOR")
      .agg(pl.len().alias("N"),
           pl.col("MONTO").sum().alias("MONTO_TOTAL"))
      .with_columns((pl.col("N") / df.height * 100).round(2).alias("PCT"))
      .sort("N", descending=True)
)
display(dist_ind)

## 7. Evolución Mensual

In [ ]:
evol = (
    df_cal.group_by(["MES_ORIGEN"])
          .agg([
              pl.len().alias("TOTAL_TRX"),
              pl.col("FLAG_FRAUDE").sum().alias("FRAUDES"),
              pl.col("MONTO").filter(pl.col("FLAG_FRAUDE") == 1).sum().alias("MONTO_FRAUDE"),
          ])
          .with_columns([
              (pl.col("FRAUDES") / pl.col("TOTAL_TRX") * 100).round(4).alias("FRAUD_RATE_PCT"),
              (pl.col("MONTO_FRAUDE") / pl.col("FRAUDES")).round(2).alias("SEVERIDAD"),
          ])
          .sort("MES_ORIGEN")
)

display(evol)

# ── Gráfico ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Evolución Mensual — Transferencias a Terceros", fontsize=13, fontweight="bold")

meses = evol["MES_ORIGEN"].to_list()

for ax, col, color, titulo in zip(
    axes,
    ["FRAUDES", "FRAUD_RATE_PCT", "MONTO_FRAUDE"],
    [COLOR_FRAUD, COLOR_ACCENT, COLOR_OK],
    ["# Fraudes", "Fraud Rate (%)", "Monto Fraude (S/)"],
):
    vals = evol[col].to_list()
    bars = ax.bar(meses, vals, color=color, alpha=0.85, edgecolor="white", linewidth=0.5)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("Mes")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.bar_label(bars, fmt="%.2f" if col == "FRAUD_RATE_PCT" else "%.0f", fontsize=8, color="white", padding=3)
    ax.grid(axis="y", alpha=0.4)

plt.tight_layout()
plt.show()

## 8. Análisis por Franja Horaria

In [ ]:
franja = (
    df_cal.group_by("FRANJA_HORARIA")
          .agg([
              pl.len().alias("TOTAL"),
              pl.col("FLAG_FRAUDE").sum().alias("FRAUDES"),
              pl.col("MONTO").filter(pl.col("FLAG_FRAUDE") == 1).sum().alias("MONTO_FRAUDE"),
          ])
          .with_columns(
              (pl.col("FRAUDES") / pl.col("TOTAL") * 100).round(3).alias("FRAUD_RATE")
          )
          .sort("FRAUD_RATE", descending=True)
)
display(franja)

fig, ax = plt.subplots(figsize=(8, 4))
franjas = franja["FRANJA_HORARIA"].to_list()
rates   = franja["FRAUD_RATE"].to_list()
colors  = [COLOR_FRAUD if r == max(rates) else COLOR_OK for r in rates]
bars = ax.barh(franjas, rates, color=colors, alpha=0.85, edgecolor="white")
ax.bar_label(bars, fmt="%.3f%%", fontsize=9, color="white", padding=4)
ax.set_title("Fraud Rate por Franja Horaria", fontsize=12, fontweight="bold")
ax.set_xlabel("Fraud Rate (%)")
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

## 9. Análisis por Hora del Día

In [ ]:
por_hora = (
    df_cal.group_by("HORA")
          .agg([
              pl.len().alias("TOTAL"),
              pl.col("FLAG_FRAUDE").sum().alias("FRAUDES"),
          ])
          .with_columns(
              (pl.col("FRAUDES") / pl.col("TOTAL") * 100).round(3).alias("FRAUD_RATE")
          )
          .sort("HORA")
)

fig, ax1 = plt.subplots(figsize=(14, 4))
horas = por_hora["HORA"].to_list()
ax1.bar(horas, por_hora["TOTAL"].to_list(), color=COLOR_OK, alpha=0.5, label="Total trx")
ax2 = ax1.twinx()
ax2.plot(horas, por_hora["FRAUD_RATE"].to_list(), color=COLOR_FRAUD,
         linewidth=2.5, marker="o", markersize=5, label="Fraud Rate")
ax1.set_xlabel("Hora del día")
ax1.set_ylabel("Volumen transacciones", color=COLOR_OK)
ax2.set_ylabel("Fraud Rate (%)", color=COLOR_FRAUD)
ax1.set_title("Volumen vs Fraud Rate por Hora — Transferencias a Terceros", fontsize=12, fontweight="bold")
ax1.set_xticks(range(0, 24))
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.show()

## 10. Distribución de Montos — Normal vs Fraude

In [ ]:
montos_f = df_cal.filter(pl.col("FLAG_FRAUDE") == 1)["MONTO"].drop_nulls().to_numpy()
montos_n = df_cal.filter(pl.col("FLAG_FRAUDE") == 0)["MONTO"].drop_nulls().to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("Distribución de Montos — Transferencias a Terceros", fontsize=12, fontweight="bold")

# Histograma superpuesto
axes[0].hist(montos_n, bins=40, color=COLOR_OK,   alpha=0.6, label="No Fraude", density=True)
axes[0].hist(montos_f, bins=40, color=COLOR_FRAUD, alpha=0.7, label="Fraude",    density=True)
axes[0].set_xlabel("Monto (S/)")
axes[0].set_ylabel("Densidad")
axes[0].set_title("Distribución (densidad)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Boxplot comparativo
axes[1].boxplot(
    [montos_n, montos_f],
    labels=["No Fraude", "Fraude"],
    patch_artist=True,
    boxprops=dict(facecolor=COLOR_OK,   alpha=0.7),
    medianprops=dict(color="white", linewidth=2),
)
axes[1].set_ylabel("Monto (S/)")
axes[1].set_title("Boxplot comparativo")
axes[1].grid(alpha=0.3)

print(f"Monto mediano Fraude    : S/ {np.median(montos_f):>10,.2f}")
print(f"Monto mediano No Fraude : S/ {np.median(montos_n):>10,.2f}")
plt.tight_layout()
plt.show()

## 11. Día de Semana vs Fraude

In [ ]:
DIAS = {1:"Lun",2:"Mar",3:"Mié",4:"Jue",5:"Vie",6:"Sáb",7:"Dom"}

por_dia = (
    df_cal.group_by("DIA_SEMANA")
          .agg([
              pl.len().alias("TOTAL"),
              pl.col("FLAG_FRAUDE").sum().alias("FRAUDES"),
          ])
          .with_columns([
              (pl.col("FRAUDES") / pl.col("TOTAL") * 100).round(3).alias("FRAUD_RATE"),
              pl.col("DIA_SEMANA").replace_strict(DIAS).alias("NOMBRE_DIA"),
          ])
          .sort("DIA_SEMANA")
)
display(por_dia)

fig, ax = plt.subplots(figsize=(9, 4))
dias_label = por_dia["NOMBRE_DIA"].to_list()
rates_dia  = por_dia["FRAUD_RATE"].to_list()
colors_dia = [COLOR_FRAUD if d in ["Sáb","Dom"] else COLOR_OK for d in dias_label]
bars = ax.bar(dias_label, rates_dia, color=colors_dia, alpha=0.85, edgecolor="white")
ax.bar_label(bars, fmt="%.3f%%", fontsize=8, color="white", padding=3)
ax.set_title("Fraud Rate por Día de Semana", fontsize=12, fontweight="bold")
ax.set_ylabel("Fraud Rate (%)")
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

## 12. Análisis de Velocidad — Cuentas de Alto Volumen

In [ ]:
vel_análisis = (
    df_cal.group_by("CUENTA")
          .agg([
              pl.len().alias("TOTAL_TRX"),
              pl.col("FLAG_FRAUDE").sum().alias("FRAUDES"),
              pl.col("MONTO").sum().alias("MONTO_TOTAL"),
              pl.col("MONTO").filter(pl.col("FLAG_FRAUDE") == 1).sum().alias("MONTO_FRAUDE"),
              pl.col("MONTO").mean().round(2).alias("TICKET_PROM"),
              pl.col("FECHA_TRX").n_unique().alias("DIAS_ACTIVO"),
          ])
          .with_columns([
              (pl.col("FRAUDES") / pl.col("TOTAL_TRX") * 100).round(3).alias("FRAUD_RATE"),
              (pl.col("TOTAL_TRX") / (pl.col("DIAS_ACTIVO") + 1)).round(2).alias("TRX_POR_DIA"),
          ])
          .filter(pl.col("FRAUDES") > 0)
          .sort("MONTO_FRAUDE", descending=True)
)

print("🔴 Top 15 cuentas con mayor monto fraudulento:")
display(vel_análisis.head(15))

## 13. Análisis de Anomalías de Monto (Z-score)

In [ ]:
print("Distribución de Z-score de monto por indicador:")
zscore_summary = (
    df_cal.group_by("INDICADOR")
          .agg([
              pl.col("ZSCORE_MONTO_CUENTA").mean().round(3).alias("Z_MEDIA"),
              pl.col("ZSCORE_MONTO_CUENTA").median().round(3).alias("Z_MEDIANA"),
              pl.col("ZSCORE_MONTO_CUENTA").max().round(3).alias("Z_MAX"),
              pl.col("FLAG_ANOMALIA_MONTO").mean().round(4).alias("PCT_ANOMALIA"),
          ])
)
display(zscore_summary)

# Scatter Z-score vs Monto
fig, ax = plt.subplots(figsize=(10, 4))
for indicador, color in zip(["F", "G", "D"], [COLOR_FRAUD, COLOR_OK, COLOR_ACCENT]):
    subset = df_cal.filter(pl.col("INDICADOR").str.strip_chars() == indicador)
    if subset.height > 0:
        ax.scatter(
            subset["ZSCORE_MONTO_CUENTA"].to_numpy(),
            subset["MONTO"].to_numpy(),
            color=color, alpha=0.4, s=15, label=indicador
        )
ax.axvline(x=2,  color="white", linestyle="--", alpha=0.5, label="z=±2")
ax.axvline(x=-2, color="white", linestyle="--", alpha=0.5)
ax.set_xlabel("Z-score de monto (por cuenta)")
ax.set_ylabel("Monto (S/)")
ax.set_title("Z-score vs Monto — Detección de Anomalías", fontsize=11, fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Propuesta de Reglas — Base para Monitor

In [ ]:
# ── Evaluar efectividad de combinaciones de flags ─────────────────────────────
reglas = [
    ("R1: Monto Alto (≥5,000)",          pl.col("FLAG_MONTO_ALTO") == 1),
    ("R2: Monto Crítico (≥10,000)",      pl.col("FLAG_MONTO_CRITICO") == 1),
    ("R3: Velocidad Alta (>3 trx/día)",  pl.col("FLAG_VEL_ALTA") == 1),
    ("R4: Anomalía Monto (|z|>2)",       pl.col("FLAG_ANOMALIA_MONTO") == 1),
    ("R5: Madrugada (0-5h)",             pl.col("FRANJA_HORARIA") == "Madrugada"),
    ("R6: Fin Semana + Monto Alto",      (pl.col("FLAG_FIN_SEMANA") == 1) & (pl.col("FLAG_MONTO_ALTO") == 1)),
    ("R7: Vel Alta + Monto Alto",        (pl.col("FLAG_VEL_ALTA") == 1) & (pl.col("FLAG_MONTO_ALTO") == 1)),
    ("R8: Anomalía + Madrugada",         (pl.col("FLAG_ANOMALIA_MONTO") == 1) & (pl.col("FRANJA_HORARIA") == "Madrugada")),
]

resultados = []
for nombre, condicion in reglas:
    sub = df_cal.filter(condicion)
    n_alertas  = sub.height
    n_fraudes  = sub.filter(pl.col("FLAG_FRAUDE") == 1).height
    precision  = n_fraudes / n_alertas * 100 if n_alertas > 0 else 0
    recall     = n_fraudes / total_fraud * 100 if total_fraud > 0 else 0
    monto_cap  = sub.filter(pl.col("FLAG_FRAUDE") == 1)["MONTO"].sum()
    resultados.append({
        "Regla"      : nombre,
        "Alertas"    : n_alertas,
        "Fraudes"    : n_fraudes,
        "Precision%" : round(precision, 2),
        "Recall%"    : round(recall, 2),
        "Monto_Cap_S/": round(monto_cap, 2) if monto_cap else 0,
    })

df_reglas = pl.DataFrame(resultados).sort("Precision%", descending=True)
print("\n📋 Evaluación de Reglas Propuestas:")
display(df_reglas)

In [ ]:
# Visualización precision vs recall por regla
fig, ax = plt.subplots(figsize=(10, 5))
regla_nombres = df_reglas["Regla"].to_list()
prec = df_reglas["Precision%"].to_list()
rec  = df_reglas["Recall%"].to_list()

x = np.arange(len(regla_nombres))
w = 0.35
ax.bar(x - w/2, prec, w, color=COLOR_FRAUD,  alpha=0.85, label="Precisión (%)", edgecolor="white")
ax.bar(x + w/2, rec,  w, color=COLOR_ACCENT, alpha=0.85, label="Recall (%)",    edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(regla_nombres, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("%")
ax.set_title("Precisión vs Recall por Regla Propuesta", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.show()

## 15. Export — Dataset enriquecido para análisis externo

In [ ]:
OUTPUT_PATH = Path(r"C:\Users\s4930359\Data_Herramientas\data\transferencias_terceros")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Parquet completo
df.write_parquet(OUTPUT_PATH / "transferencias_terceros_enriquecido.parquet")

# Excel solo fraudes para revisión
(
    df_cal
    .filter(pl.col("FLAG_FRAUDE") == 1)
    .sort("MONTO", descending=True)
    .to_pandas()
    .to_excel(OUTPUT_PATH / "fraudes_transferencias_terceros.xlsx", index=False)
)

# Tabla de reglas
df_reglas.to_pandas().to_excel(OUTPUT_PATH / "evaluacion_reglas.xlsx", index=False)

print("✅ Archivos exportados:")
print(f"   → transferencias_terceros_enriquecido.parquet")
print(f"   → fraudes_transferencias_terceros.xlsx")
print(f"   → evaluacion_reglas.xlsx")

---
## 📝 Resumen del Análisis

| Dimensión | Hallazgos clave |
|---|---|
| **Volumen** | Ver KPIs sección 6 |
| **Temporalidad** | Revisar hora pico de fraude (sección 9) |
| **Montos** | Fraude tiende a montos más altos (sección 10) |
| **Velocidad** | Cuentas con >3 trx/día concentran riesgo |
| **Reglas** | Ver tabla de precisión/recall (sección 14) |

> **Próximo paso sugerido:** Implementar en Monitor las reglas R6 y R7 (mayor precisión compuesta)  
> y monitorear variación mes a mes del Fraud Rate en transferencias a terceros.